# Run 322 artifact probe

Draft notebook for `run_id=322b4485-557c-434b-b2e1-0776d15a515b`.

This notebook loads the pinned BTCUSDT `1h` price artifacts and the two signal matrices used by the run:

- `ma.dema` from `signals/1h/ma.dema/signals.i8.npy`
- `ma.ema` from `signals/1h/ma.ema/signals.i8.npy`

Pinned runtime identity:

- slot: `slot_a`
- generation: `1`
- asof_date: `2026-04-02`
- manifest_hash: `13df35a144a9706b6b40c949f71ddc5dd23d60da10bbda81c12e26f9676faaae`
- timeframe: `1h`
- symbol: `BTCUSDT`


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import yaml

RUN_ID = "322b4485-557c-434b-b2e1-0776d15a515b"
TIMEFRAME = "1h"
RUN_TIME_RANGE_START = datetime(2017, 10, 6, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_END = datetime(2026, 3, 31, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_START_MS = int(RUN_TIME_RANGE_START.timestamp() * 1000)
RUN_TIME_RANGE_END_MS = int(RUN_TIME_RANGE_END.timestamp() * 1000)

ARTIFACT_ROOT = Path("/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a")
PRICE_DIR = ARTIFACT_ROOT / "prices" / TIMEFRAME
DEMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.dema"
EMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.ema"

SLOT_MANIFEST_PATH = ARTIFACT_ROOT / "manifest.yaml"
PRICE_OPEN_TIME_PATH = PRICE_DIR / "open_time.i64.npy"
PRICE_CLOSE_TIME_PATH = PRICE_DIR / "close_time.i64.npy"
PRICE_OHLCV_PATH = PRICE_DIR / "ohlcv.f32.npy"
DEMA_MANIFEST_PATH = DEMA_DIR / "manifest.yaml"
EMA_MANIFEST_PATH = EMA_DIR / "manifest.yaml"
DEMA_SIGNAL_PATH = DEMA_DIR / "signals.i8.npy"
EMA_SIGNAL_PATH = EMA_DIR / "signals.i8.npy"

REQUEST_INDICATOR_GRIDS = [
    {
        "indicator_id": "ma.dema",
        "sources": ["close"],
        "window_range": [5, 200],
    },
    {
        "indicator_id": "ma.ema",
        "sources": ["high", "ohlc4"],
        "window_range": [5, 200],
    },
]

for path in [
    SLOT_MANIFEST_PATH,
    PRICE_OPEN_TIME_PATH,
    PRICE_CLOSE_TIME_PATH,
    PRICE_OHLCV_PATH,
    DEMA_MANIFEST_PATH,
    EMA_MANIFEST_PATH,
    DEMA_SIGNAL_PATH,
    EMA_SIGNAL_PATH,
]:
    print(f"{path}: exists={path.exists()}")


In [ ]:
slot_manifest = yaml.safe_load(SLOT_MANIFEST_PATH.read_text())
dema_manifest = yaml.safe_load(DEMA_MANIFEST_PATH.read_text())
ema_manifest = yaml.safe_load(EMA_MANIFEST_PATH.read_text())

prices_by_timeframe = {item["timeframe"]: item for item in slot_manifest["prices"]}
price_manifest = prices_by_timeframe[TIMEFRAME]

print("slot:", slot_manifest["slot"], "generation:", slot_manifest["slot_generation"], "asof_date:", slot_manifest["asof_date"])
print("price bars:", price_manifest["coverage"]["bar_count"])
print("dema signals shape:", tuple(dema_manifest["signals"]["shape"]))
print("ema signals shape:", tuple(ema_manifest["signals"]["shape"]))
print("request indicator grids:", REQUEST_INDICATOR_GRIDS)


In [ ]:
price_open_time = np.load(PRICE_OPEN_TIME_PATH, mmap_mode="r")
price_close_time = np.load(PRICE_CLOSE_TIME_PATH, mmap_mode="r")
price_ohlcv = np.load(PRICE_OHLCV_PATH, mmap_mode="r")
dema_signals = np.load(DEMA_SIGNAL_PATH, mmap_mode="r")
ema_signals = np.load(EMA_SIGNAL_PATH, mmap_mode="r")

print("price_open_time:", price_open_time.shape, price_open_time.dtype)
print("price_close_time:", price_close_time.shape, price_close_time.dtype)
print("price_ohlcv:", price_ohlcv.shape, price_ohlcv.dtype)
print("dema_signals:", dema_signals.shape, dema_signals.dtype)
print("ema_signals:", ema_signals.shape, ema_signals.dtype)


In [ ]:
open_time_index = price_open_time.astype("datetime64[ms]")
close_time_index = price_close_time.astype("datetime64[ms]")
time_mask = (price_open_time >= RUN_TIME_RANGE_START_MS) & (price_open_time <= RUN_TIME_RANGE_END_MS)

print("bars inside run time range:", int(time_mask.sum()))
print("first matching bar:", np.datetime_as_string(open_time_index[time_mask][0], timezone="UTC"))
print("last matching bar:", np.datetime_as_string(open_time_index[time_mask][-1], timezone="UTC"))


In [ ]:
sample_indices = np.flatnonzero(time_mask)[:5]
sample_prices = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index[idx], timezone="UTC"),
        "open": float(price_ohlcv[idx, 0]),
        "high": float(price_ohlcv[idx, 1]),
        "low": float(price_ohlcv[idx, 2]),
        "close": float(price_ohlcv[idx, 3]),
        "volume": float(price_ohlcv[idx, 4]),
    }
    for idx in sample_indices
]
sample_prices


In [ ]:
signal_probe = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "dema_row_0": int(dema_signals[0, idx]),
        "ema_row_0": int(ema_signals[0, idx]),
    }
    for idx in range(12)
]
signal_probe


## Notes

- The two `signals.i8.npy` files above are the exact artifact-backed matrices used by the run.
- Each file stores the full indicator matrix for one indicator family on the pinned `1h` timeline.
- The run-specific subset is narrower than the full matrix:
  - `ma.dema`: `source=close`, `window=5..200`
  - `ma.ema`: `source in {high, ohlc4}`, `window=5..200`
- This draft notebook intentionally stops at loading the exact pinned `npy` artifacts and probing timeline alignment.
- Next extension: reconstruct the row mapping for the selected sources/windows from the v2 signal rules/defaults catalog and then slice the relevant rows from each matrix.


## Additional 15m artifact loads for BTCUSDT spot

This section adds a broader `15m` probe on the same pinned slot for three extra indicators from different families:

- `ma.sma`
- `momentum.roc`
- `volatility.stddev`

It also loads:

- `prices/15m/*`
- derived price calculations (`open`, `high`, `low`, `close`, `hlc3`, `ohlc4`)
- strict `hit_times/1m/*` TP/SL tables and levels

The existing `1h` run-specific experiment above is kept intact.


In [ ]:
TIMEFRAME_15M = "15m"
PRICE_DIR_15M = ARTIFACT_ROOT / "prices" / TIMEFRAME_15M
HIT_TIMES_DIR_1M = ARTIFACT_ROOT / "hit_times" / "1m"

EXTRA_INDICATORS_15M = {
    "ma.sma": {
        "family": "ma",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "momentum.roc": {
        "family": "momentum",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": [5, 7, 10, 14, 21, 28, 42, 63, 84, 126],
    },
    "volatility.stddev": {
        "family": "volatility",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": [10, 14, 20, 28, 42, 56, 84, 126],
        "signal_defaults": {"long_delta_periods": -5, "short_delta_periods": -10},
    },
}

PRICE_OPEN_TIME_15M_PATH = PRICE_DIR_15M / "open_time.i64.npy"
PRICE_CLOSE_TIME_15M_PATH = PRICE_DIR_15M / "close_time.i64.npy"
PRICE_OHLCV_15M_PATH = PRICE_DIR_15M / "ohlcv.f32.npy"
HIT_TIMES_MANIFEST_PATH = HIT_TIMES_DIR_1M / "manifest.yaml"
TP_VALUES_PATH = HIT_TIMES_DIR_1M / "tp_values.f32.npy"
SL_VALUES_PATH = HIT_TIMES_DIR_1M / "sl_values.f32.npy"
LONG_TP_PATH = HIT_TIMES_DIR_1M / "long_tp.u32.npy"
SHORT_TP_PATH = HIT_TIMES_DIR_1M / "short_tp.u32.npy"
LONG_SL_PATH = HIT_TIMES_DIR_1M / "long_sl.u32.npy"
SHORT_SL_PATH = HIT_TIMES_DIR_1M / "short_sl.u32.npy"

EXTRA_SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "signals.i8.npy",
    }
    for indicator_id in EXTRA_INDICATORS_15M
}

paths_to_check = [
    PRICE_OPEN_TIME_15M_PATH,
    PRICE_CLOSE_TIME_15M_PATH,
    PRICE_OHLCV_15M_PATH,
    HIT_TIMES_MANIFEST_PATH,
    TP_VALUES_PATH,
    SL_VALUES_PATH,
    LONG_TP_PATH,
    SHORT_TP_PATH,
    LONG_SL_PATH,
    SHORT_SL_PATH,
] + [value for item in EXTRA_SIGNAL_PATHS_15M.values() for value in item.values()]

for path in paths_to_check:
    print(f"{path}: exists={path.exists()}")


In [ ]:
price_manifest_15m = prices_by_timeframe[TIMEFRAME_15M]
signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in EXTRA_SIGNAL_PATHS_15M.items()
}
hit_times_manifest = yaml.safe_load(HIT_TIMES_MANIFEST_PATH.read_text())

print("15m price bars:", price_manifest_15m["coverage"]["bar_count"])
for indicator_id, doc in signal_manifests_15m.items():
    print(
        indicator_id,
        "rows_count=", doc["rows_count"],
        "shape=", tuple(doc["signals"]["shape"]),
        "signal_defaults=", doc["grid"].get("signals_v1_params_defaults", {}),
    )
print("hit_times tables:", hit_times_manifest["tables"])
print("tp grid path:", hit_times_manifest["tp_values"])
print("sl grid path:", hit_times_manifest["sl_values"])


In [ ]:
price_open_time_15m = np.load(PRICE_OPEN_TIME_15M_PATH, mmap_mode="r")
price_close_time_15m = np.load(PRICE_CLOSE_TIME_15M_PATH, mmap_mode="r")
price_ohlcv_15m = np.load(PRICE_OHLCV_15M_PATH, mmap_mode="r")

signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r")
    for indicator_id, paths in EXTRA_SIGNAL_PATHS_15M.items()
}

tp_values_1m = np.load(TP_VALUES_PATH, mmap_mode="r")
sl_values_1m = np.load(SL_VALUES_PATH, mmap_mode="r")
long_tp_1m = np.load(LONG_TP_PATH, mmap_mode="r")
short_tp_1m = np.load(SHORT_TP_PATH, mmap_mode="r")
long_sl_1m = np.load(LONG_SL_PATH, mmap_mode="r")
short_sl_1m = np.load(SHORT_SL_PATH, mmap_mode="r")

print("price_open_time_15m:", price_open_time_15m.shape, price_open_time_15m.dtype)
print("price_close_time_15m:", price_close_time_15m.shape, price_close_time_15m.dtype)
print("price_ohlcv_15m:", price_ohlcv_15m.shape, price_ohlcv_15m.dtype)
for indicator_id, matrix in signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)
print("tp_values_1m:", tp_values_1m.shape, tp_values_1m.dtype)
print("sl_values_1m:", sl_values_1m.shape, sl_values_1m.dtype)
print("long_tp_1m:", long_tp_1m.shape, long_tp_1m.dtype)
print("short_tp_1m:", short_tp_1m.shape, short_tp_1m.dtype)
print("long_sl_1m:", long_sl_1m.shape, long_sl_1m.dtype)
print("short_sl_1m:", short_sl_1m.shape, short_sl_1m.dtype)


In [ ]:
open_time_index_15m = price_open_time_15m.astype("datetime64[ms]")
close_time_index_15m = price_close_time_15m.astype("datetime64[ms]")
time_mask_15m = (price_open_time_15m >= RUN_TIME_RANGE_START_MS) & (price_open_time_15m <= RUN_TIME_RANGE_END_MS)

price_fields_15m = {
    "open": np.asarray(price_ohlcv_15m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_15m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_15m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_15m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_15m[:, 4], dtype=np.float32),
}
price_fields_15m["hlc3"] = (price_fields_15m["high"] + price_fields_15m["low"] + price_fields_15m["close"]) / 3.0
price_fields_15m["ohlc4"] = (price_fields_15m["open"] + price_fields_15m["high"] + price_fields_15m["low"] + price_fields_15m["close"]) / 4.0

print("15m bars inside run time range:", int(time_mask_15m.sum()))
print("first 15m bar:", np.datetime_as_string(open_time_index_15m[time_mask_15m][0], timezone="UTC"))
print("last 15m bar:", np.datetime_as_string(open_time_index_15m[time_mask_15m][-1], timezone="UTC"))
print("derived price fields:", tuple(price_fields_15m.keys()))


In [ ]:
def build_source_blocks(matrix, *, sources, param_values):
    block = len(param_values)
    return {
        source: matrix[idx * block:(idx + 1) * block]
        for idx, source in enumerate(sources)
    }

signal_source_blocks_15m = {
    indicator_id: build_source_blocks(
        signal_matrices_15m[indicator_id],
        sources=meta["sources"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in EXTRA_INDICATORS_15M.items()
}

for indicator_id, source_map in signal_source_blocks_15m.items():
    print("indicator:", indicator_id)
    for source_name, block in source_map.items():
        print("  ", source_name, block.shape, block.dtype)


In [ ]:
sample_indices_15m = np.flatnonzero(time_mask_15m)[:5]
sample_prices_15m = [
    {
        "open_time": np.datetime_as_string(open_time_index_15m[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index_15m[idx], timezone="UTC"),
        "open": float(price_fields_15m["open"][idx]),
        "high": float(price_fields_15m["high"][idx]),
        "low": float(price_fields_15m["low"][idx]),
        "close": float(price_fields_15m["close"][idx]),
        "hlc3": float(price_fields_15m["hlc3"][idx]),
        "ohlc4": float(price_fields_15m["ohlc4"][idx]),
        "volume": float(price_fields_15m["volume"][idx]),
    }
    for idx in sample_indices_15m
]
sample_prices_15m


In [ ]:
signal_probe_15m = []
for idx in np.flatnonzero(time_mask_15m)[:8]:
    signal_probe_15m.append({
        "open_time": np.datetime_as_string(open_time_index_15m[idx], timezone="UTC"),
        "ma.sma.close.row_0": int(signal_source_blocks_15m["ma.sma"]["close"][0, idx]),
        "ma.sma.open.row_0": int(signal_source_blocks_15m["ma.sma"]["open"][0, idx]),
        "momentum.roc.low.row_0": int(signal_source_blocks_15m["momentum.roc"]["low"][0, idx]),
        "momentum.roc.ohlc4.row_0": int(signal_source_blocks_15m["momentum.roc"]["ohlc4"][0, idx]),
        "volatility.stddev.high.row_0": int(signal_source_blocks_15m["volatility.stddev"]["high"][0, idx]),
        "volatility.stddev.close.row_0": int(signal_source_blocks_15m["volatility.stddev"]["close"][0, idx]),
    })
signal_probe_15m
